In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F
import re

In [0]:
dbutils.widgets.text("repo_urls","","Repository URLs")
dbutils.widgets.text("analysis_window","1y","Analysis Window")
repo_urls = dbutils.widgets.get("repo_urls")
analysis_window = dbutils.widgets.get("analysis_window")

if not repo_urls:
    raise ValueError("No repository URLs provided.")

repo_urls = [url.strip() for url in repo_urls.split(",")]

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import Row

today = datetime.utcnow()
if analysis_window == "1m":
    analysis_since = today - timedelta(days=30)
elif analysis_window == "2m":
    analysis_since = today - timedelta(days=60)
elif analysis_window == "3m":
    analysis_since = today - timedelta(days=90)
elif analysis_window == "6m":
    analysis_since = today - timedelta(days=180)
elif analysis_window == "1y":
    analysis_since = today - timedelta(days=365)
elif analysis_window == "2y":
    analysis_since = today - timedelta(days=730)
else:
    analysis_since = today - timedelta(days=365)



config = [
    Row(
        analysis_window=analysis_window,
        analysis_since=analysis_since.isoformat()
    )
]

df = spark.createDataFrame(config)

df.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.bronze.pipeline_config")

In [0]:
repositories = []

for i, url in enumerate(repo_urls, start=1):
    match = re.search(r"github\.com/([^/]+)/([^/]+)", url)
    if not match:
        raise ValueError(f"Invalid GitHub URL: {url}")

    owner = match.group(1)
    repository = match.group(2)

    repositories.append(Row(repository_id=i,owner=owner,repository_name=repository,repo_full_name=f"{owner}/{repository}",repository_url=url))

In [0]:
config_df = spark.createDataFrame(repositories)
config_df.write.format("delta").mode("overwrite").saveAsTable("gitobservatory.bronze.repository_config")

In [0]:
%sql
SELECT * FROM gitobservatory.bronze.repository_config

repository_id,owner,repository_name,repo_full_name,repository_url
1,dbt-labs,dbt-core,dbt-labs/dbt-core,https://github.com/dbt-labs/dbt-core
2,PrefectHQ,prefect,PrefectHQ/prefect,https://github.com/PrefectHQ/prefect
